In [14]:
!pip install langchain langchain-core langchain-community \
langchain-google-genai google-generativeai \
faiss-cpu pypdf sentence-transformers langdetect \
fastapi uvicorn python-telegram-bot nest-asyncio
!pip install -U langchain-text-splitters


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Import successful ✅")

Import successful ✅


In [ ]:
api_key="GOOGLE_API_KEY"
import os
import google.generativeai as genai
#load the secret key
api_key="API_KEY"
#configure the generative ai model
genai.configure(api_key=api_key)

In [17]:
for model in genai.list_models():#models we can use,
  if "generateContent" in model.supported_generation_methods:#only text generation models
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [18]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "Python Activity Pseudocode.pdf"

loader = PyPDFLoader(file_path)
documents = loader.load()

print("Pages loaded:", len(documents))

Pages loaded: 5


In [19]:
#Split the PDF into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Chunks created:", len(chunks))

Chunks created: 12


In [20]:
#Create Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model loaded ✅


In [21]:
#import nece. libraries for doc. load and text splitting
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [22]:
 file_path= "Python Activity Pseudocode.pdf"

In [23]:
loader=PyPDFLoader(file_path)#loader is like a reciever, will give pdf to documents

In [24]:
documents= loader.load()

In [25]:
print(f"loaded {len(documents)} pages from the pdf")

loaded 5 pages from the pdf


In [26]:
print(documents) #holds with metadata

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2025-07-02T00:58:57+05:30', 'author': 'Saarah Saeed', 'moddate': '2025-07-02T00:58:57+05:30', 'source': 'Python Activity Pseudocode.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='Python Activity Pseudocode \n  \nPseudocode for question -1 \n1. Initialize an empty dictionary: \n   students = {} \n \n2. Repeat until user chooses to exit: \n   Display menu: \n      1. Register a student \n      2. Enroll in a course \n      3. View all students \n      4. Search student by roll number \n      5. Remove course from a student \n      6. Exit \n \n   Read user choice \n \n3. If choice == 1 (Register a student): \n   Prompt for student name and roll number \n   If roll number already exists: \n       Show "Already registered" message \n   Else: \n       Add to dictionary: \n           students[roll] = { "name": name, "courses": [] } \n       Show success message \n

In [27]:
print("the content of the first page:")
print(documents[0].page_content[:500])#500 tokens

the content of the first page:
Python Activity Pseudocode 
  
Pseudocode for question -1 
1. Initialize an empty dictionary: 
   students = {} 
 
2. Repeat until user chooses to exit: 
   Display menu: 
      1. Register a student 
      2. Enroll in a course 
      3. View all students 
      4. Search student by roll number 
      5. Remove course from a student 
      6. Exit 
 
   Read user choice 
 
3. If choice == 1 (Register a student): 
   Prompt for student name and roll number 
   If roll number already exists: 
   


In [ ]:
#create an instance of the text_splitter

In [28]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50 ) #genereal ratio 10:1 100->10
#split the loader docs into the final chunks
chunks=text_splitter.split_documents(documents)
print(f"split into {len(chunks)} chunks")

split into 12 chunks


In [29]:
print("some content from the first chunk")
print(chunks[0].page_content)

some content from the first chunk
Python Activity Pseudocode 
  
Pseudocode for question -1 
1. Initialize an empty dictionary: 
   students = {} 
 
2. Repeat until user chooses to exit: 
   Display menu: 
      1. Register a student 
      2. Enroll in a course 
      3. View all students 
      4. Search student by roll number 
      5. Remove course from a student 
      6. Exit 
 
   Read user choice 
 
3. If choice == 1 (Register a student): 
   Prompt for student name and roll number 
   If roll number already exists:


In [ ]:
#own data , relevant chunks (divide)

In [30]:
#importing the necessary libraries for embedding and vector store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [31]:
model_name="sentence-transformers/all-MiniLM-L6-v2"
embeddings=HuggingFaceEmbeddings(model_name=model_name)
print("Local embedding model loaded !")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Local embedding model loaded !


#Creating a Vector Store- The Brain!

In [32]:
vectorstore= FAISS.from_documents(chunks,embeddings)
vectorstore.save_local("faiss_index")
print("The Vector Store is built and saved to the 'FAISS index' ")

The Vector Store is built and saved to the 'FAISS index' 


In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI #llm
from langchain.prompts import PromptTemplate #prompt template
from langchain.chains import create_retrieval_chain #chain for retrieval
from langchain.chains.combine_documents import create_stuff_documents_chain #

ModuleNotFoundError: No module named 'langchain.prompts'

In [34]:
#load faiss vector store
vectorstore=FAISS.load_local("faiss_index",embeddings, allow_dangerous_deserialization= True)

In [47]:
#initializing the LLM
llm=ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash", # Corrected model name
    temperature=0.7,
    google_api_key=api_key
 )#  (0-1)less , model more precise answer.High, model got more creative independence, 0.7=ideal
print("Component Loaded")

Component Loaded


In [48]:
#GPT
response = llm.invoke("What is Python?")
print(response.content)

Python is a **high-level, interpreted, general-purpose programming language** known for its readability and simplicity.

Let's break down what that means and why it's so popular:

### Key Characteristics:

1.  **High-Level:**
    *   It abstracts away many complex details of computer hardware, making it much easier for humans to read, write, and understand code compared to low-level languages like C or Assembly.
    *   You don't need to worry about memory management or CPU registers.

2.  **Interpreted:**
    *   Unlike compiled languages (like C++ or Java), Python code is executed line by line by an interpreter at runtime, rather than being fully translated into machine code before execution.
    *   **Pros:** Faster development cycle (no separate compilation step), easier debugging.
    *   **Cons:** Generally slower execution speed than compiled languages.

3.  **General-Purpose:**
    *   This means it's not specialized for one particular type of application. You can use Python fo

In [49]:
docs = retriever.invoke("How do I register a student?")

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
Answer only using the provided context.

Context:
{context}

Question:
How do I register a student?
"""

response = llm.invoke(prompt)

print(response.content)

To register a student:
1.  Choose option `1. Register a student` from the menu.
2.  Prompt for the student's name and roll number.
3.  If the roll number already exists, it will show "Already registered".
4.  Otherwise, the student will be added to the `students` dictionary with their name and an empty list for courses, and a success message will be shown.


In [50]:
def ask_rag(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
    Answer using only the provided context.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content

In [51]:
print(
    ask_rag(
        "How do I register a student?"
    )
)

To register a student:
1.  Choose option `1. Register a student` from the menu.
2.  Prompt for the student's name and roll number.
3.  If the roll number already exists, display "Already registered".
4.  Otherwise, add the student to the `students` dictionary as `students[roll] = { "name": name, "courses": [] }`.
5.  Show a success message.


In [56]:
answer = ask_rag("What happens if a roll number already exists?")

print(answer)

If a roll number already exists, it will Show "Already registered" message.


1.3.4


In [57]:
while True:
    question = input("\nAsk a question (or type 'exit'): ")

    if question.lower() == "exit":
        break

    print("\nAnswer:")
    print(ask_rag(question))


Ask a question (or type 'exit'): How do I register a student?

Answer:
To register a student:
1. Choose option `1. Register a student` from the menu.
2. Prompt for the student's name and roll number.
3. If the roll number already exists, display "Already registered".
4. Else, add the student to the `students` dictionary with their name and an empty list for courses, then show a success message.

Ask a question (or type 'exit'): exit


In [36]:
#Defining the prompt template
prompt_template="""
Answer the question as thoroughly as possible based on the provided context.
If you dont know the answer then act a little cheezy and give a funny answer by teasing the person.
Make a conversation engaging and relatable by observing the person.

context:
{context}

Question: {input}

Answer:
"""
prompt=PromptTemplate(template=prompt_template,input_variables=["context","input"])

NameError: name 'PromptTemplate' is not defined

In [37]:
#operating on LLM and Prompt
#create the document chain
document_chain=create_stuff_documents_chain(llm,prompt)
#create the retriever
retriever=vectorstore.as_retriever()
#retriever chain
retriever_chain=create_retrieval_chain(retriever, document_chain)

NameError: name 'create_stuff_documents_chain' is not defined

In [38]:
question="Tell me about Space Science"
response= retriever_chain.invoke({"input":question})

print("Question:")
print(question)

print("\n--Answer--")
print(response["answer"])

NameError: name 'retriever_chain' is not defined

In [39]:
#import some necessary tools for multilingual conversations
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2" #model for multi lingual
embeddings_multi=HuggingFaceEmbeddings(model_name=model_name) #object
vectorstore_multi= FAISS.from_documents(chunks, embeddings_multi)
vectorstore_multi.save_local("faiss_index_multilingual")#saving
print("new vector store is created")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

new vector store is created


In [40]:
#for the final pipeline for our multilingual model
from langdetect import detect
print("Language detection model tool imported")

Language detection model tool imported


In [41]:
#1load the multilingual vector store
vectorstore_multi=FAISS.load_local("faiss_index_multilingual",embeddings_multi, allow_dangerous_deserialization= True)

In [42]:
#2- create a new prompt template that include a language instruction
prompt_template_multi="""
Answer the question as thoroughly as possible based on the provided context.
If you don't know the answer , just say I don't have time for your extra bullshit, don't try to make up an answer.
Your final answer must be in the following langusage:(language)

Context:
{context}
Question: {input}

Answer:
"""

prompt_multi=PromptTemplate(
    template=prompt_template_multi,
    input_variables=["context","input","language"]


)

NameError: name 'PromptTemplate' is not defined

In [43]:
# 3- Recreate the retrieval chain for the new components
retrieval_multi=vectorstore_multi.as_retriever()
document_chain_multi=create_stuff_documents_chain(llm, prompt_multi)
retrieval_chain_multi=create_retrieval_chain(retrieval_multi, document_chain)

NameError: name 'create_stuff_documents_chain' is not defined

In [ ]:
question_spanish="Cuéntame sobre el código de Python"
detect_language=detect(question_spanish)

In [44]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

docs = retriever.invoke("What is Python?")

print("Retrieved docs:", len(docs))
print(docs[0].page_content[:200])

Retrieved docs: 3
Python Activity Pseudocode 
  
Pseudocode for question -1 
1. Initialize an empty dictionary: 
   students = {} 
 
2. Repeat until user chooses to exit: 
   Display menu: 
      1. Register a student 


In [46]:
for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [45]:
response = llm.invoke("What is Python?")

print(response.content)

ChatGoogleGenerativeAIError: Error calling model 'models/gemini-2.5-flash-preview-05-20' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-2.5-flash-preview-05-20 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
detect_language

In [ ]:
response= retrieval_chain_multi.invoke({
    "input" : question_spanish,
    "language": detect_language
})
print(question_spanish)
print("\n------Answer------")
print(response["answer"])

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain


In [ ]:
# memory object which conatin the memory
memory= ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")

# new powerfull conversation chain
conversational_chian=ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retrieval_multi,
    memory=memory,
    return_source_documents=True #this is the key for getting sources back
)

#API

In [ ]:
# MORE FUNCTIONALITY
#convo based on historical data/context


In [ ]:
#We are adding pyngrok to our installation {cannot run on local host on colab environ.we need external environment/server  }
!pip install -q pyngrok

In [ ]:
# Import all necc libraries for this session
import nest_asyncio #nest_asyncio → Fixes a bug in Colab so we can run web servers inside it.
from fastapi import FastAPI #FastAPI → Makes a web app for your chatbot (like creating “doors” called endpoints).
from pydantic import BaseModel, Field #Pydantic (BaseModel) → Decides what kind of data can come in (example: only text questions, not random junk).
import uvicorn #uvicorn → The engine that runs your FastAPI app (so it’s live on the web).
import asyncio #asyncio → Helps handle many requests at the same time without blocking.
from pyngrok import ngrok #pyngrok → Creates a public link so others can use your chatbot (since Colab only runs locally).
from google.colab import userdata #google.colab.userdata → Safe place to keep your secret API keys in Colab.
import threading #threading → Runs the web server in the background while you keep using your notebook.

print("All necessary libraries for API and tunneling are imported")

#refer too notebok

In [ ]:
# defining our data models
# updating the base models from new conversational chain
class Source(BaseModel):
  content: str= Field(description="the actual text snippet from the source document")
  page: int| None= Field(description="the page number of the source doc., if available") #we will use field for better documentation
class QueryRequest(BaseModel):
  question: str
  language: str | None='en'
class QueryResponse(BaseModel):
  answer: str =Field(description="the generated answer from the chatbot")
  sources: list[Source]


#creating a fast api instance and apply the patch
app=FastAPI()
nest_asyncio.apply()#for multiple loops

print("app instance created")


In [ ]:

#end point that will recieve the question
@app.post("/query", response_model=QueryResponse)
async def process_query(request: QueryRequest):
  print(f"Recieved query: '{request.questio}' in language: '{request.language}' ")

  response= conversational_chain.invoke({
      "input": request.question,
      "language": request.language
  })

  #format the source docs. for the response
  formatted_sources=[]
  if response.get("source_documents"):
    for doc in response["source_documents"]:
      source= Source(content=doc.page_content, page=doc.metadata.get("page"))
      formatted_sources.append(source)

  return QueryResponse(answer=response["answer"], sources= formatted_source)



  #A simple root endpoint to check if server is running

  @app.post("/clear_memory")
  async def clear_memory():
    memory.clear()
    return {"message":"Memory has been cleared"}

  @app.get("/")
  async def read_root():
    return {"message": "Campus API is running"}

    print("api endpoints are defined")


more functionality ?

In [ ]:
# lets run the server and create a public url with ngrok cuz we are using collab

# set up the ngrok auth token
NGROK_AUTHTOKEN= userdata.get("NGROK_AUTH")
ngrok.set_auth_token(NGROK_AUTHTOKEN)

#define a func to  run a uvicorn server

def run_server():
  uvicorn.run(app, host="0.0.0.0", port= 8000)
# running the server in the saperate thread

server_thread= threading.thread(target=run_server)
server_thread.start()

# having buffer time to let the server start

import time
time.sleep(5)

# create a public url using ngrok
public_url= ngrok.connect(8000)

In [ ]:
#working on its goldfish memory today
# 1- Coversation Buffer Memory (tool from langchain)
# 2- Reliability(will tell the source , from where it has searced or extracted the answer)
#for it , chage retrieval chain to conversational retrieval chain
#change base model.
# change endpoints


In [ ]:
#(for the prev.notebook, refer to w  )

# How to test the full exp

In [ ]:
#How to test the full experience
# VS ------
#